In [1]:
import pandas as pd
import os
import pyarrow.parquet as pq
from PIL import Image
from io import BytesIO

train_df = pd.read_csv("metadata/train.csv")
val_df = pd.read_csv("metadata/val.csv")
test_df = pd.read_csv("metadata/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(train_df.head())

Train: (90256, 4)
Validation: (10923, 4)
Test: (10941, 4)


,shard,row_index,patient_id,pneumonia
0,train-00000-of-00006.parquet,0,1,0
1,train-00000-of-00006.parquet,1,1,0
2,train-00000-of-00006.parquet,2,1,0
3,train-00000-of-00006.parquet,3,2,0
4,train-00000-of-00006.parquet,4,3,0


In [8]:
from src.dataset import ChestXrayDataset, RepeatChannels


In [13]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
import timm
import torch.nn as nn

model = timm.create_model(
    "vit_small_patch16_224.augreg_in1k",
    pretrained=True,
    num_classes=14 #automatically replace the pretrained classification head with a new linear layer producing 14 outputs
)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters()) / 1e6,
    "million"
)

from timm.data import resolve_model_data_config

data_config = resolve_model_data_config(model)
print(data_config)

Parameters: 21.671054 million
{'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.9, 'crop_mode': 'center'}


In [9]:
from torchvision import transforms

vit_transform = transforms.Compose([
    transforms.Resize(
        (224, 224),
        interpolation=transforms.InterpolationMode.BICUBIC
    ),
    transforms.ToTensor(),
    RepeatChannels(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [10]:
train_dataset = ChestXrayDataset(
    dataframe=train_df,
    data_dir='data/data/',
    transform=vit_transform,
    image_mode="L"
)

In [11]:
image, target = train_dataset[0]

print("Image shape:", image.shape)
print("Image range:", image.min().item(), image.max().item())
print("Target shape:", target.shape)

Image shape: torch.Size([3, 224, 224])
Image range: -1.0 1.0
Target shape: torch.Size([14])


In [14]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.05
)

num_epochs = 5

In [15]:
from torch.utils.data import DataLoader
from src.train import train, RowGroupBatchSampler

batch_sampler = RowGroupBatchSampler(
    train_dataset,
    batch_size=32
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=batch_sampler,
    num_workers=0,
    pin_memory=True,
    #persistent_workers=True
)


In [19]:
model.train()
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Labels:", labels)
print("Image dtype:", images.dtype)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 14])
Labels: tensor([[0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
 

In [21]:
optimizer.zero_grad(set_to_none=True)

with torch.autocast(
    device_type=device.type,
    enabled=(device.type == "cuda")
):
    logits = model(images)
    loss = criterion(logits, labels)

loss.backward()
optimizer.step()

print("Loss:", loss.item())
print("Output shape:", logits.shape)

if device.type == "cuda":
    print(
        "Peak GPU memory:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB"
    )

Loss: 0.803658127784729
Output shape: torch.Size([32, 14])
Peak GPU memory: 2.21 GB


In [23]:
val_dataset = ChestXrayDataset(
    dataframe=val_df,
    data_dir='data/data/',
    transform=vit_transform,
    image_mode="L"
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [24]:
model = train(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=5,
    print_every=100,
    checkpoint_dir="checkpoints",
    run_name="vit_s16_adamw_clean"
)

Epoch 1 | Batch 100/2832 | Train loss: 0.1815 | Interval: 0.9 min | Elapsed: 0.9 min | ETA: 25.5 min
Epoch 1 | Batch 200/2832 | Train loss: 0.1788 | Interval: 0.9 min | Elapsed: 1.9 min | ETA: 24.8 min
Epoch 1 | Batch 300/2832 | Train loss: 0.1743 | Interval: 1.0 min | Elapsed: 2.9 min | ETA: 24.1 min
Epoch 1 | Batch 400/2832 | Train loss: 0.1717 | Interval: 1.0 min | Elapsed: 3.8 min | ETA: 23.2 min
Epoch 1 | Batch 500/2832 | Train loss: 0.1706 | Interval: 1.0 min | Elapsed: 4.8 min | ETA: 22.4 min
Epoch 1 | Batch 600/2832 | Train loss: 0.1692 | Interval: 1.0 min | Elapsed: 5.8 min | ETA: 21.5 min
Epoch 1 | Batch 700/2832 | Train loss: 0.1684 | Interval: 1.0 min | Elapsed: 6.8 min | ETA: 20.6 min
Epoch 1 | Batch 800/2832 | Train loss: 0.1689 | Interval: 1.0 min | Elapsed: 7.8 min | ETA: 19.7 min
Epoch 1 | Batch 900/2832 | Train loss: 0.1681 | Interval: 1.0 min | Elapsed: 8.8 min | ETA: 18.8 min
Epoch 1 | Batch 1000/2832 | Train loss: 0.1675 | Interval: 1.0 min | Elapsed: 9.8 min | ETA

KeyboardInterrupt: 